In [ ]:
# tvp_var_connectedness.py
# 목적:
# - merged_var_input.csv 기반 TVP-VAR connectedness 추정
# - 시점별 GFEVD, TCI, TO, FROM, NET 계산
# - 결과 csv / png 저장
#
# 입력:
#   ./merged_var_input.csv
#
# 주요 변수:
#   dlog_SOLVPN, dlog_COPPER, dlog_DXY, d_UST10Y, dlog_VIX
#
# 출력:
#   ./tvp_var_out/
#       tvpvar_coefficients.csv
#       tvpvar_covariances.csv
#       gfevd_last.csv
#       connectedness_timevarying.csv
#       connectedness_summary_last.csv
#       tci_plot.png
#       net_spillover_plot.png
#       gfevd_last_heatmap.png

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

warnings.filterwarnings("ignore")

# =========================================================
# 0) Config
# =========================================================
INPUT_CSV = "./merged_var_input.csv"
OUT_DIR = "./tvp_var_out"
os.makedirs(OUT_DIR, exist_ok=True)

DATE_COL = "Date"

VAR_COLS = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_DXY",
    "d_UST10Y",
    "dlog_VIX"
]

LAG_ORDER = 1
HORIZON = 10

# forgetting factors
LAMBDA_BETA = 0.99   # coefficient update
LAMBDA_SIGMA = 0.99  # covariance update

# initial training length
INIT_WINDOW = 150

# numerical stability
RIDGE = 1e-6
EPS = 1e-10

# =========================================================
# 1) Load
# =========================================================
df = pd.read_csv(INPUT_CSV)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

missing = [c for c in VAR_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[[DATE_COL] + VAR_COLS].dropna().reset_index(drop=True)

print("Loaded shape:", df.shape)
print("Date range:", df[DATE_COL].min(), "~", df[DATE_COL].max())
print("Variables:", VAR_COLS)

Y_raw = df[VAR_COLS].values.astype(float)
dates = df[DATE_COL].values
T_full, k = Y_raw.shape

if T_full <= INIT_WINDOW + LAG_ORDER + 30:
    raise ValueError("Not enough observations for TVP-VAR estimation.")

# =========================================================
# 2) Helper functions
# =========================================================
def build_var_xy(Y, p):
    """
    Y: (T, k)
    return:
      Y_target: (T-p, k)
      X_lag:    (T-p, 1 + k*p)   # const 포함
    """
    T, k = Y.shape
    Ys = []
    Xs = []
    for t in range(p, T):
        y_t = Y[t]
        x_t = [1.0]
        for lag in range(1, p + 1):
            x_t.extend(Y[t - lag].tolist())
        Ys.append(y_t)
        Xs.append(x_t)
    return np.asarray(Ys), np.asarray(Xs)

def companion_from_beta(beta_mat, k, p):
    """
    beta_mat: shape (1 + k*p, k)
    return companion A of shape (k*p, k*p)
    """
    A = np.zeros((k * p, k * p))
    lag_block = beta_mat[1:, :].T  # (k, k*p)
    A[:k, :k*p] = lag_block
    if p > 1:
        A[k:, :-k] = np.eye(k * (p - 1))
    return A

def compute_ma_matrices(A, H, k, p):
    """
    companion matrix 기반 MA coefficient 계산
    Phi_0 ... Phi_{H-1}
    each Phi_h: (k, k)
    """
    kp = k * p
    J = np.zeros((k, kp))
    J[:, :k] = np.eye(k)

    Phi_list = []
    A_power = np.eye(kp)
    for h in range(H):
        Phi_h = J @ A_power @ J.T
        Phi_list.append(Phi_h)
        A_power = A_power @ A
    return Phi_list

def generalized_fevd(A, Sigma, H, k, p):
    """
    Pesaran-Shin generalized FEVD
    return: (k, k), row-normalized
    """
    Sigma = np.asarray(Sigma, dtype=float)
    Sigma = 0.5 * (Sigma + Sigma.T)
    Sigma = Sigma + np.eye(k) * RIDGE

    Phi_list = compute_ma_matrices(A, H, k, p)

    num = np.zeros((k, k))
    den = np.zeros(k)

    for h in range(H):
        Phi_h = Phi_list[h]
        temp = Phi_h @ Sigma @ Phi_h.T
        den += np.diag(temp)

        for j in range(k):
            e_j = np.zeros(k)
            e_j[j] = 1.0
            sigma_jj = Sigma[j, j]
            if sigma_jj <= EPS:
                sigma_jj = EPS
            contrib = (Phi_h @ Sigma @ e_j) ** 2 / sigma_jj
            num[:, j] += contrib

    fevd = np.zeros((k, k))
    for i in range(k):
        if den[i] <= EPS:
            continue
        fevd[i, :] = num[i, :] / den[i]

    row_sums = fevd.sum(axis=1, keepdims=True)
    row_sums[row_sums <= EPS] = 1.0
    fevd = fevd / row_sums
    return fevd

def connectedness_from_fevd(fevd, var_names):
    """
    fevd[i,j] = variable i forecast error variance explained by shock j
    row sum = 1
    """
    k = fevd.shape[0]
    off_diag_sum = fevd.sum() - np.trace(fevd)
    tci = 100.0 * off_diag_sum / k

    from_others = 100.0 * (fevd.sum(axis=1) - np.diag(fevd))
    to_others = 100.0 * (fevd.sum(axis=0) - np.diag(fevd))
    net = to_others - from_others

    out = pd.DataFrame({
        "Variable": var_names,
        "TO": to_others,
        "FROM": from_others,
        "NET": net
    })
    return tci, out

# =========================================================
# 3) Prepare lagged data
# =========================================================
Y_tgt, X_lag = build_var_xy(Y_raw, LAG_ORDER)
dates_eff = dates[LAG_ORDER:]
T = len(Y_tgt)
m = X_lag.shape[1]  # 1 + k*p

print("Effective T:", T)
print("Regressor dim per equation:", m)

# =========================================================
# 4) Initial OLS for starting values
# =========================================================
init_end = INIT_WINDOW
Y_init = Y_tgt[:init_end]
X_init = X_lag[:init_end]

B0 = np.linalg.inv(X_init.T @ X_init + RIDGE * np.eye(m)) @ (X_init.T @ Y_init)   # (m, k)
E0 = Y_init - X_init @ B0
S0 = (E0.T @ E0) / max(len(E0) - m, 1)
S0 = 0.5 * (S0 + S0.T) + RIDGE * np.eye(k)

# equation-by-equation RLS states
beta_list = [B0[:, i].copy() for i in range(k)]
P_list = [100.0 * np.eye(m) for _ in range(k)]

Sigma_t = S0.copy()

# outputs
beta_records = []
sigma_records = []
conn_records = []

last_fevd = None
last_conn = None

# =========================================================
# 5) TVP-VAR recursion
# =========================================================
for t in range(init_end, T):
    x_t = X_lag[t]          # (m,)
    y_t = Y_tgt[t]          # (k,)
    date_t = pd.to_datetime(dates_eff[t])

    y_hat = np.zeros(k)
    residual = np.zeros(k)

    # ---- coefficient update: equation-by-equation RLS ----
    for i in range(k):
        beta_prev = beta_list[i]
        P_prev = P_list[i]

        # predict step with forgetting factor
        P_pred = P_prev / LAMBDA_BETA

        xPx = x_t @ P_pred @ x_t
        gain = (P_pred @ x_t) / (1.0 + xPx)

        y_pred_i = x_t @ beta_prev
        err_i = y_t[i] - y_pred_i

        beta_new = beta_prev + gain * err_i
        P_new = P_pred - np.outer(gain, x_t) @ P_pred

        beta_list[i] = beta_new
        P_list[i] = P_new

        y_hat[i] = y_pred_i
        residual[i] = err_i

    # ---- covariance update ----
    Sigma_t = LAMBDA_SIGMA * Sigma_t + (1.0 - LAMBDA_SIGMA) * np.outer(residual, residual)
    Sigma_t = 0.5 * (Sigma_t + Sigma_t.T) + RIDGE * np.eye(k)

    # beta matrix shape (m, k)
    B_t = np.column_stack(beta_list)

    # companion matrix
    A_t = companion_from_beta(B_t, k=k, p=LAG_ORDER)

    # ---- GFEVD / connectedness ----
    fevd_t = generalized_fevd(A=A_t, Sigma=Sigma_t, H=HORIZON, k=k, p=LAG_ORDER)
    tci_t, conn_df_t = connectedness_from_fevd(fevd_t, VAR_COLS)

    # records: coefficients
    row_beta = {"Date": date_t}
    for eq_i, eq_name in enumerate(VAR_COLS):
        row_beta[f"{eq_name}_const"] = B_t[0, eq_i]
        for lag in range(1, LAG_ORDER + 1):
            for var_j, var_name in enumerate(VAR_COLS):
                idx = 1 + (lag - 1) * k + var_j
                row_beta[f"{eq_name}_L{lag}_{var_name}"] = B_t[idx, eq_i]
    beta_records.append(row_beta)

    # records: covariances
    row_sigma = {"Date": date_t}
    for i, vi in enumerate(VAR_COLS):
        for j, vj in enumerate(VAR_COLS):
            row_sigma[f"Sigma_{vi}_{vj}"] = Sigma_t[i, j]
    sigma_records.append(row_sigma)

    # records: connectedness
    row_conn = {"Date": date_t, "TCI": tci_t}
    for _, r in conn_df_t.iterrows():
        v = r["Variable"]
        row_conn[f"TO_{v}"] = r["TO"]
        row_conn[f"FROM_{v}"] = r["FROM"]
        row_conn[f"NET_{v}"] = r["NET"]
    conn_records.append(row_conn)

    last_fevd = fevd_t.copy()
    last_conn = conn_df_t.copy()

# =========================================================
# 6) Save outputs
# =========================================================
beta_df = pd.DataFrame(beta_records)
sigma_df = pd.DataFrame(sigma_records)
conn_df = pd.DataFrame(conn_records)

beta_df.to_csv(os.path.join(OUT_DIR, "tvpvar_coefficients.csv"), index=False, encoding="utf-8-sig")
sigma_df.to_csv(os.path.join(OUT_DIR, "tvpvar_covariances.csv"), index=False, encoding="utf-8-sig")
conn_df.to_csv(os.path.join(OUT_DIR, "connectedness_timevarying.csv"), index=False, encoding="utf-8-sig")

if last_fevd is not None:
    gfevd_last_df = pd.DataFrame(last_fevd, index=VAR_COLS, columns=VAR_COLS)
    gfevd_last_df.to_csv(os.path.join(OUT_DIR, "gfevd_last.csv"), encoding="utf-8-sig")

if last_conn is not None:
    last_conn.to_csv(os.path.join(OUT_DIR, "connectedness_summary_last.csv"), index=False, encoding="utf-8-sig")

# =========================================================
# 7) Plot: TCI
# =========================================================
plt.figure(figsize=(14, 5))
plt.plot(pd.to_datetime(conn_df["Date"]), conn_df["TCI"])
plt.title(f"TVP-VAR Total Connectedness Index (H={HORIZON}, p={LAG_ORDER})")
plt.xlabel("Date")
plt.ylabel("TCI")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "tci_plot.png"), dpi=300)
plt.close()

# =========================================================
# 8) Plot: NET spillover
# =========================================================
plt.figure(figsize=(14, 7))
for v in VAR_COLS:
    plt.plot(pd.to_datetime(conn_df["Date"]), conn_df[f"NET_{v}"], label=v)

plt.axhline(0.0, linestyle="--")
plt.title("TVP-VAR Net Spillover")
plt.xlabel("Date")
plt.ylabel("NET")
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "net_spillover_plot.png"), dpi=300)
plt.close()

# =========================================================
# 9) Plot: last-date GFEVD heatmap with numeric annotations
# =========================================================
if last_fevd is not None:
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(last_fevd, aspect="auto")

    ax.set_xticks(np.arange(k))
    ax.set_yticks(np.arange(k))
    ax.set_xticklabels(VAR_COLS, rotation=30, ha="right")
    ax.set_yticklabels(VAR_COLS)

    for i in range(k):
        for j in range(k):
            ax.text(j, i, f"{last_fevd[i, j]:.2f}", ha="center", va="center")

    ax.set_title("Last-date GFEVD Heatmap")
    ax.set_xlabel("Shock Source")
    ax.set_ylabel("Response Variable")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "gfevd_last_heatmap.png"), dpi=300)
    plt.close()

# =========================================================
# 10) Console summary
# =========================================================
print("\nSaved to:", OUT_DIR)
print(" - tvpvar_coefficients.csv")
print(" - tvpvar_covariances.csv")
print(" - connectedness_timevarying.csv")
print(" - gfevd_last.csv")
print(" - connectedness_summary_last.csv")
print(" - tci_plot.png")
print(" - net_spillover_plot.png")
print(" - gfevd_last_heatmap.png")

if last_conn is not None:
    print("\nLast-date connectedness summary:")
    print(last_conn)

if not conn_df.empty:
    print("\nTCI summary:")
    print(conn_df["TCI"].describe())

Loaded shape: (1325, 6)
Date range: 2020-10-13 00:00:00 ~ 2026-01-12 00:00:00
Variables: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_DXY', 'd_UST10Y', 'dlog_VIX']
Effective T: 1324
Regressor dim per equation: 6

Saved to: ./tvp_var_out
 - tvpvar_coefficients.csv
 - tvpvar_covariances.csv
 - connectedness_timevarying.csv
 - gfevd_last.csv
 - connectedness_summary_last.csv
 - tci_plot.png
 - net_spillover_plot.png
 - gfevd_last_heatmap.png

Last-date connectedness summary:
      Variable         TO       FROM        NET
0  dlog_SOLVPN  20.070798  18.445867   1.624931
1  dlog_COPPER   9.471769  21.859273 -12.387504
2     dlog_DXY  18.047477   2.158302  15.889175
3     d_UST10Y   2.942358   6.389069  -3.446711
4     dlog_VIX  21.877919  23.557810  -1.679891

TCI summary:
count    1174.000000
mean       16.857048
std         2.845213
min        12.219437
25%        15.039658
50%        16.127520
75%        17.976451
max        29.656432
Name: TCI, dtype: float64


: 